# MIW CWFS Intrinsic Check (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-07-13
**Last Modified:** 2026-07-13
**Status:** In Progress
**Keywords:** MIW, intrinsic wavefront, CWFS, IntrinsicZernikes, OCS, CCS, AOS validation

## Description

The Measured Intrinsic Wavefront (MIW) went live in the AOS.  This notebook
verifies that the **correct intrinsic values were used** for the 4 corner
wavefront sensors (CWFS) on regular (science) images, by independently
recomputing the expected intrinsic from **my own calibration source tables** and
comparing to the `zk_intrinsic_*` stored in the live aggregate tables.

The live path (traced in `ts_intrinsic_wavefront`): the AOS applies
`lsst.ip.isr.IntrinsicZernikes(table=<per-detector CCS>, table_ocs=<OCS>)` and
stores, per donut,
`calib.getIntrinsicZernikes(field_x, field_y, rotTelPos=...)`.
So the check rebuilds that exact calib object from the OCS + per-detector CCS
source parquets, evaluates it at each CWFS donut's field position and the
visit rotator, and differences against the aggregate.

Because a handful of frame conventions (field-argument order, rotator
sign/offset, and which of the aggregate's OCS/CCS/NW intrinsic columns
`getIntrinsicZernikes` reproduces) are not obvious a priori, the notebook
**auto-solves** that small discrete space against the aggregate: the config that
reproduces the stored intrinsic (correlation ~1, slope ~1, tiny residual)
confirms the correct values were used *and* tells us the convention; if **no**
config matches, that is the alarm (wrong / stale calib ingested, or a bug).

A separate validation section plots the OCS and CCS focal-plane maps (as in
`aos_miw_ocs_ccs_maps.ipynb`) as a further sanity check on the maps behind the
calib.

**Output:** `output/miw_cwfs_intrinsic_check.parquet` (per donut x Noll:
expected vs aggregate + residual) and the comparison / map figures.

**References:** `ts_intrinsic_wavefront` `calib_tables.py`,
`bin.src/run_make_calib_tables.py`, `bin.src/ingest_calib_tables.py`
(`IntrinsicZernikes(table=ccs, table_ocs=ocs)` /
`getIntrinsicZernikes(field_x, field_y, rotTelPos)`); `aos_miw_ocs_ccs_maps.ipynb`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-13 | Aaron Roodman | Initial version — CWFS intrinsic verification vs live aggregate + OCS/CCS map validation |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Load calib source tables -> IntrinsicZernikes](#calib)
4. [Validation: OCS / CCS focal-plane maps](#maps)
5. [Load aggregate CWFS donuts (last night)](#agg)
6. [Recompute expected intrinsic + auto-solve convention](#recompute)
7. [Comparison plots + PASS/FAIL](#compare)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters
# ============================================================
from datetime import date, timedelta

# ---- Summit Butler + the live aggregate for last night's regular images ----
# Butler access mirrors rubin-work/olr/code/nightly_table.py (Summit *embargo*
# butler; makeDefaultButler resolves the LSSTCam/runs/quickLook chain), keyed by
# exposure.day_obs.  But use the **Raw** aggregate -- one row per donut pair, so
# we see the actual intrinsic used per donut (not the per-corner Avg).
USE_SUMMIT_BUTLER = True                         # makeDefaultButler("LSSTCam", embargo=True)
BUTLER_REPO  = "LSSTCam"                         # fallback only (if USE_SUMMIT_BUTLER=False)
COLLECTIONS  = ["LSSTCam/runs/quickLook"]        # fallback only
DATASET_TYPE = "aggregateAOSVisitTableRaw"       # per-donut-pair rows
DAY_OBS      = int((date.today() - timedelta(days=1)).strftime("%Y%m%d"))  # last night
MAX_SEQ      = None      # None = all visits; or an int to load only the first N seq_num (fast peek)

# ---- My own calib SOURCE tables (mirrors the USDF rubin-data layout) ----
CALIB_DIR = "~/notebooks/rubin-data/aos/intrinsic_zernikes/v2_ajr"   # OCS + per-detector CCS
OCS_FILE  = "intrinsic_aberrations_OCS.parquet" # per-instrument OCS table (whole map)
# per-detector CCS tables expected as  intrinsic_aberrations_CCS_det<NNN>.parquet

# ---- MIW maps for the OCS/CCS validation plots (set None to skip that section) ----
MAPS_PATH = "~/notebooks/rubin-data/aos/intrinsic_zernikes/intrinsic_split_maps.parquet"

# ---- CWFS detectors: the Avg table carries the 4 SW0 corners (as in olr) ----
CWFS = {191: "R00_SW0", 195: "R04_SW0", 199: "R40_SW0", 203: "R44_SW0"}

KEY_NOLL      = [4, 5, 6, 7, 8, 11]             # Noll shown in the scatter panel
TOL_UM        = 0.01                             # PASS if max median|expected-aggregate| <= this (µm)
OUT_PARQUET   = "output/miw_cwfs_intrinsic_check.parquet"
FP_RADIUS_DEG = 1.75
PCT           = 98                               # map color percentile

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from tqdm import tqdm                              # plain text bar (no ipywidgets)

from lsst.daf.butler import Butler
from lsst.ip.isr import IntrinsicZernikes

if USE_SUMMIT_BUTLER:                            # same as olr/nightly_table.py
    from lsst.summit.utils import butlerUtils
    butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=True)
else:
    butler = Butler(BUTLER_REPO, collections=COLLECTIONS)
NAME2ID = {v: k for k, v in CWFS.items()}


def to_deg(a):
    """Angles to degrees (aggregate positions may be stored in radians)."""
    a = np.asarray(a, float)
    return np.degrees(a) if np.nanmax(np.abs(a)) < 0.2 else a

<a id='calib'></a>
## 3. Load calib source tables -> IntrinsicZernikes

Reconstructs the exact `IntrinsicZernikes(table=ccs, table_ocs=ocs)` the AOS ingested, one per CWFS detector, from **my own** source parquets.

In [ ]:
# Rebuild the exact calib the AOS ingested: shared OCS + per-detector CCS.
cdir = Path(CALIB_DIR).expanduser()
ocs = Table.read(str(cdir / OCS_FILE), format="parquet")
calib_by_det, missing = {}, []
for det in CWFS:
    p = cdir / f"intrinsic_aberrations_CCS_det{det:03d}.parquet"
    if p.exists():
        calib_by_det[det] = IntrinsicZernikes(table=Table.read(str(p), format="parquet"),
                                              table_ocs=ocs)
    else:
        missing.append(det)
if missing:
    print(f"WARNING: no CCS table for detectors {missing} (copy them, or drop from CWFS)")
assert calib_by_det, "no CWFS calibs loaded -- check CALIB_DIR / OCS_FILE / CCS filenames"

calib_noll = [int(x) for x in np.asarray(next(iter(calib_by_det.values())).noll_indices)]
print(f"loaded {len(calib_by_det)} CWFS calibs: {sorted(calib_by_det)}")
print(f"calib Noll (n={len(calib_noll)}): {calib_noll}")

<a id='maps'></a>
## 4. Validation: OCS / CCS focal-plane maps

A visual sanity check on the maps behind the calib (as in `aos_miw_ocs_ccs_maps.ipynb`): telescope-fixed OCS (left) and camera-fixed CCS (right) per Zernike, with the CWFS footprint centroids marked (+).

In [ ]:
# OCS (telescope-fixed) | CCS (camera field + per-CCD height) focal-plane maps.
if MAPS_PATH:
    tmap = Table.read(str(Path(MAPS_PATH).expanduser()), format="parquet")
    mthx = np.asarray(tmap["thx_deg"], float)
    mthy = np.asarray(tmap["thy_deg"], float)
    mjs = sorted({int(m.group(1)) for c in tmap.colnames
                  for m in [re.match(r"Z(\d+)_OCS$", c)] if m})
    # CWFS footprint centroids (CCS frame) to overlay where the corners sit
    corner_xy = {CWFS[d]: (float(np.mean(calib_by_det[d].field_x)),
                           float(np.mean(calib_by_det[d].field_y)))
                 for d in calib_by_det}

    def plot_map(ax, vals, title, vlim, mark=False, cmap="RdBu_r"):
        v = np.asarray(vals, float); fin = np.isfinite(v)
        tcf = ax.tricontourf(mthx[fin], mthy[fin], v[fin],
                             levels=np.linspace(-vlim, vlim, 21), cmap=cmap, extend="both")
        ax.add_patch(plt.Circle((0, 0), FP_RADIUS_DEG, fill=False, ec="k", lw=0.6, alpha=0.4))
        if mark:
            for nm, (cx, cy) in corner_xy.items():
                ax.plot(cx, cy, "k+", ms=9, mew=1.4)
        ax.set_aspect("equal")
        ax.set_xlim(-FP_RADIUS_DEG, FP_RADIUS_DEG); ax.set_ylim(-FP_RADIUS_DEG, FP_RADIUS_DEG)
        ax.set_title(title, fontsize=9); ax.set_xlabel("thx [deg]"); ax.set_ylabel("thy [deg]")
        return tcf

    def vlim_for(*arrs):
        vv = np.concatenate([np.asarray(a, float)[np.isfinite(a)] for a in arrs])
        return max(float(np.nanpercentile(np.abs(vv), PCT)), 1e-6) if vv.size else 1.0

    print(f"maps: {len(mthx)} field points; Noll {mjs}")
    for j in mjs:
        O = np.asarray(tmap[f"Z{j}_OCS"], float)
        C = np.asarray(tmap[f"Z{j}_CCS"], float)
        vl = vlim_for(O, C)
        fig, axs = plt.subplots(1, 2, figsize=(11, 4.6), layout="constrained")
        tcf = plot_map(axs[0], O, f"Z{j}  OCS (telescope)", vl)
        plot_map(axs[1], C, f"Z{j}  CCS (camera) + CWFS(+)", vl, mark=True)
        fig.colorbar(tcf, ax=axs, shrink=0.85, label="µm")
        fig.suptitle(f"MIW validation — Z{j}", fontsize=12)
        plt.show()
else:
    print("MAPS_PATH is None -- skipping OCS/CCS map validation")

### 4b. Z4 CCS at full per-CCD resolution

The per-detector CCS calib tables sample Z4 (camera field + CCD height) on a fine grid within each CCD. Pool every detector's footprint points and scatter them across the focal plane (as in `run_psf_fp_maps`) to see the full-resolution per-CCD height structure — much finer than the smooth disk-grid maps above.

In [ ]:
# Z4 CCS at FULL per-CCD resolution: every per-detector CCS calib table carries
# its footprint grid of points (fine sampling within each CCD) with Z4 = camera
# field + per-point CCD height.  Pool them all and scatter across the focal plane
# (same per-CCD focal-plane scatter style as run_psf_fp_maps) to reveal the fine
# per-CCD height structure.  Uses the calib tables (all CCDs if you built the full
# set); with only the 4 CWFS tables it shows just the corners.
import glob
ccs_files = sorted(glob.glob(str(cdir / "intrinsic_aberrations_CCS_det*.parquet")))
xs, ys, z4 = [], [], []
for f in ccs_files:
    tt = Table.read(f, format="parquet")
    if "Z4" in tt.colnames and {"x", "y"} <= set(tt.colnames):
        xs.append(np.asarray(tt["x"], float))
        ys.append(np.asarray(tt["y"], float))
        z4.append(np.asarray(tt["Z4"], float))
if xs:
    xs, ys, z4 = np.concatenate(xs), np.concatenate(ys), np.concatenate(z4)
    fin = np.isfinite(z4)
    v = float(np.nanpercentile(np.abs(z4[fin]), 99)) or 1e-3
    fig, ax = plt.subplots(figsize=(7.8, 7.0), layout="constrained")
    ax.add_patch(plt.Circle((0, 0), FP_RADIUS_DEG, fill=False, ec="k", lw=0.6, alpha=0.5))
    sc = ax.scatter(xs[fin], ys[fin], c=z4[fin], s=4, cmap="RdBu_r",
                    vmin=-v, vmax=v, edgecolors="none")
    ax.set_aspect("equal")
    ax.set_xlim(-1.02 * FP_RADIUS_DEG, 1.02 * FP_RADIUS_DEG)
    ax.set_ylim(-1.02 * FP_RADIUS_DEG, 1.02 * FP_RADIUS_DEG)
    ax.set_xlabel("FP x [deg]"); ax.set_ylabel("FP y [deg]")
    ax.set_title(f"Z4 CCS full-resolution — {len(ccs_files)} detectors, "
                 f"{len(xs)} footprint points")
    fig.colorbar(sc, ax=ax, shrink=0.85, label="Z4 CCS [µm]")
    plt.show()
    print(f"Z4 CCS: {len(ccs_files)} detectors, {len(xs)} footprint points "
          f"(fine per-CCD sampling from the calib tables)")
else:
    print("no per-detector CCS calib tables found in CALIB_DIR for the full-res Z4 map")

<a id='agg'></a>
## 5. Load aggregate CWFS donuts (last night)

The live **Raw** aggregate for regular images (one row per donut pair, so we see the intrinsic used per donut), via the Summit embargo butler keyed by `exposure.day_obs` (same butler as `rubin-work/olr/code/nightly_table.py`, but the Raw dataset rather than Avg). Keep the CWFS rows with their field positions and the stored `zk_intrinsic_*`.

In [ ]:
# Pull last night's per-donut-pair aggregates (Raw) and keep the CWFS rows.
refs = list(butler.query_datasets(DATASET_TYPE, where=f"exposure.day_obs = {DAY_OBS}"))
refs.sort(key=lambda r: int(r.dataId["visit"]))     # by seq_num order
if MAX_SEQ:                                          # fast peek: first N seq_num only
    refs = refs[:int(MAX_SEQ)]
    print(f"MAX_SEQ={MAX_SEQ}: limiting to the first {len(refs)} visits")
print(f"loading {len(refs)} {DATASET_TYPE} datasets on day_obs={DAY_OBS}")

FRAME_COLS = ("OCS", "CCS", "NW")
rows, agg_noll, shown = [], None, False
for ref in tqdm(refs, desc="Butler aggregate"):
    t = butler.get(ref)
    if not shown:                                # report the schema of the live table once
        print("aggregate columns:", list(t.colnames))
        print("meta keys:", list(t.meta))
        print("detectors present:", sorted(set(str(x) for x in t["detector"])))
        shown = True
    if agg_noll is None and t.meta.get("nollIndices") is not None:
        agg_noll = [int(x) for x in t.meta["nollIndices"]]
    det_raw = np.array([str(x) for x in t["detector"]])
    ids = np.array([NAME2ID.get(d, int(d) if d.lstrip("-").isdigit() else -1)
                    for d in det_raw])
    keep = np.isin(ids, list(calib_by_det))
    if not keep.any():
        continue
    sub = t[keep]
    # field position: getIntrinsicZernikes wants the CCS-frame (camera) angle;
    # prefer thx_CCS/thy_CCS, fall back to OCS positions (flagged).
    have_ccs = "thx_CCS" in t.colnames and "thy_CCS" in t.colnames
    fxc, fyc = ("thx_CCS", "thy_CCS") if have_ccs else ("thx_OCS", "thy_OCS")
    zkI = {f: np.asarray(sub[f"zk_intrinsic_{f}"], float)
           for f in FRAME_COLS if f"zk_intrinsic_{f}" in t.colnames}
    rows.append(dict(
        visit=int(ref.dataId["visit"]),
        seq=int(ref.dataId["visit"] - DAY_OBS * 100000),
        det_id=ids[keep], ccs_frame=have_ccs,
        thx_CCS=to_deg(sub[fxc]), thy_CCS=to_deg(sub[fyc]),
        thx_OCS=to_deg(sub["thx_OCS"]) if "thx_OCS" in t.colnames else None,
        thy_OCS=to_deg(sub["thy_OCS"]) if "thy_OCS" in t.colnames else None,
        zkI=zkI))

assert rows, "no CWFS rows found -- check DATASET_TYPE / DAY_OBS / detector names"
frames = [f for f in FRAME_COLS if all(f in r["zkI"] for r in rows)]
if not all(r["ccs_frame"] for r in rows):
    print("WARNING: no thx_CCS/thy_CCS in the aggregate -> using OCS positions as the "
          "field input; the rotator search should still land, but confirm the frame.")
print(f"{len(rows)} visits with CWFS rows; aggregate Noll (n={len(agg_noll)}): {agg_noll}")
print(f"intrinsic frames present: {frames}")

<a id='recompute'></a>
## 6. Recompute expected intrinsic + auto-solve convention

For each CWFS donut, `getIntrinsicZernikes(field_x, field_y, rotTelPos)`. The field-argument order, rotator sign/offset, and comparison frame are auto-solved against the aggregate (best mean correlation). A match (slope~1, tiny residual) confirms the correct values were used and reveals the convention; no match => alarm.

In [ ]:
# Common Noll between calib and aggregate; column indexers into each.
common = [j for j in agg_noll if j in calib_noll]
ci = [calib_noll.index(j) for j in common]      # -> calib getIntrinsicZernikes columns
ai = [agg_noll.index(j) for j in common]        # -> aggregate zk_intrinsic columns
print(f"comparing {len(common)} shared Noll: {common}")


def visit_rot(r):
    """Rotator (deg) implied by the aggregate's own CCS->OCS position rotation."""
    if r["thx_OCS"] is None:
        return 0.0
    aC = np.arctan2(r["thy_CCS"], r["thx_CCS"])
    aO = np.arctan2(r["thy_OCS"], r["thx_OCS"])
    d = aO - aC
    return float(np.degrees(np.arctan2(np.nanmean(np.sin(d)), np.nanmean(np.cos(d)))))


def eval_visit(r, field_order, rot):
    """Expected intrinsic at this visit's CWFS donuts, shared-Noll columns."""
    fxn, fyn = field_order
    exp = np.full((len(r["det_id"]), len(calib_noll)), np.nan)
    for det in np.unique(r["det_id"]):
        m = r["det_id"] == det
        vals = calib_by_det[int(det)].getIntrinsicZernikes(
            field_x=np.asarray(r[fxn][m], float),
            field_y=np.asarray(r[fyn][m], float),
            rotTelPos=float(rot))
        exp[m] = np.asarray(vals)
    return exp[:, ci]


# ---- auto-solve the convention space on a sample of visits ----
FIELD_OPTS = [("thx_CCS", "thy_CCS"), ("thy_CCS", "thx_CCS")]
ROT_VARIANTS = [(s, o) for s in (+1, -1) for o in (0.0, 90.0, 180.0, 270.0)]
sample = rows[:min(6, len(rows))]

best = None      # (score, field_order, sign, offset, frame)
for fo in FIELD_OPTS:
    for (s, o) in ROT_VARIANTS:
        E = np.vstack([eval_visit(r, fo, s * visit_rot(r) + o) for r in sample])
        for f in frames:
            A = np.vstack([r["zkI"][f][:, ai] for r in sample])
            cs = []
            for p in range(len(common)):
                x, y = E[:, p], A[:, p]
                m = np.isfinite(x) & np.isfinite(y)
                if m.sum() > 3 and np.std(x[m]) > 0 and np.std(y[m]) > 0:
                    cs.append(abs(np.corrcoef(x[m], y[m])[0, 1]))
            score = float(np.mean(cs)) if cs else 0.0
            if best is None or score > best[0]:
                best = (score, fo, s, o, f)

score, FO, S, O, FRAME = best
print(f"best convention: field={FO}, rotTelPos={S:+d}*rot_from_positions+{O:g}, "
      f"compare-frame={FRAME}  (mean|corr|={score:.3f})")

# ---- apply best config to ALL visits -> tidy comparison ----
recs = []
for r in rows:
    rot = S * visit_rot(r) + O
    E = eval_visit(r, FO, rot)
    A = r["zkI"][FRAME][:, ai]
    for k, det in enumerate(r["det_id"]):
        for p, j in enumerate(common):
            recs.append(dict(visit=r["visit"], corner=CWFS[int(det)], j=int(j),
                             expected=E[k, p], aggregate=A[k, p],
                             thx_CCS=r["thx_CCS"][k], thy_CCS=r["thy_CCS"][k], rot_deg=rot))
cmp = pd.DataFrame(recs)
cmp["resid"] = cmp["expected"] - cmp["aggregate"]   # bracket: .aggregate is a DataFrame method
Path(OUT_PARQUET).parent.mkdir(parents=True, exist_ok=True)
cmp.to_parquet(OUT_PARQUET)

fin = np.isfinite(cmp["expected"]) & np.isfinite(cmp["aggregate"])
sl, off = np.polyfit(cmp["aggregate"][fin], cmp["expected"][fin], 1)
print(f"expected = {sl:.4f} * aggregate {off:+.4f}   (slope~1, offset~0 => values match; "
      f"slope~1e6/1e-6 => a unit mismatch)")
print(f"wrote {OUT_PARQUET}  ({len(cmp)} rows)")

<a id='compare'></a>
## 7. Comparison plots + PASS/FAIL

In [ ]:
# ---- PASS/FAIL: max median |residual| over corner x Noll ----
gg = (cmp.dropna(subset=["resid"]).groupby(["corner", "j"]).resid
        .apply(lambda x: float(np.median(np.abs(x)))))
max_med = float(gg.max())
print(f"max median|expected-aggregate| over corner x Noll = {max_med:.4f} µm  "
      f"(TOL={TOL_UM})  ->  {'PASS' if max_med <= TOL_UM else 'FAIL'}")
print("\nworst (corner, Noll):")
print(gg.sort_values(ascending=False).head(10).to_string())

# ---- scatter: expected vs aggregate for the key Noll, colored by corner ----
jshow = [j for j in KEY_NOLL if j in common]
corners = sorted(cmp.corner.unique())
cmapc = {c: plt.cm.tab10(i % 10) for i, c in enumerate(corners)}
ncol = 3; nrow = int(np.ceil(len(jshow) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.8 * nrow),
                        layout="constrained", squeeze=False)
for ax, j in zip(axs.ravel(), jshow):
    d = cmp[(cmp.j == j)].dropna(subset=["expected", "aggregate"])
    for c in corners:
        dc = d[d.corner == c]
        ax.scatter(dc["aggregate"], dc["expected"], s=10, alpha=0.6, color=cmapc[c], label=c)
    if len(d):
        lo = float(min(d["aggregate"].min(), d["expected"].min()))
        hi = float(max(d["aggregate"].max(), d["expected"].max()))
        ax.plot([lo, hi], [lo, hi], "k:", lw=0.8)
        rms = float(np.sqrt(np.nanmean((d["expected"] - d["aggregate"]) ** 2)))
        ax.set_title(f"Z{j}   rms={rms:.4f} µm", fontsize=9)
    ax.set_xlabel("aggregate zk_intrinsic [µm]"); ax.set_ylabel("expected (calib) [µm]")
    ax.grid(alpha=0.3)
for ax in axs.ravel()[len(jshow):]:
    ax.axis("off")
axs.ravel()[0].legend(fontsize=7, loc="best")
fig.suptitle(f"CWFS expected (calib) vs aggregate intrinsic — day_obs {DAY_OBS} "
             f"[frame {FRAME}]", fontsize=12)
plt.show()

# ---- heatmap: median |residual| per (corner, Noll) ----
piv = (cmp.dropna(subset=["resid"]).assign(absr=lambda d: d.resid.abs())
         .pivot_table(index="corner", columns="j", values="absr", aggfunc="median"))
fig, ax = plt.subplots(figsize=(1.1 + 0.45 * piv.shape[1], 1.0 + 0.45 * piv.shape[0]),
                       layout="constrained")
im = ax.imshow(piv.values, aspect="auto", cmap="magma_r",
               vmin=0, vmax=max(TOL_UM, float(np.nanpercentile(piv.values, 95))))
ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels([f"Z{j}" for j in piv.columns], fontsize=7)
ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=8)
fig.colorbar(im, ax=ax, label="median |resid| [µm]")
ax.set_title(f"median |expected-aggregate| (TOL={TOL_UM} µm)", fontsize=10)
plt.show()